# Numerai Classic v6.0 ("Unity") — MiniRocket Regressor (`NumeraiExtractions` + `MiniRocketNumeraiEstimator`)

- **Data Loading (`data_utils.load_unity_numerai_data_cached`)**: Resolves and downloads the Unity v6.0 Parquets using the official Kaggle clients (`kagglehub.dataset_download` and `kaggle.api.kaggle_api_extended.KaggleApi`) with `numerapi.NumerAPI` fallback, caches processed Parquets, and supports training-only downsampling (`downsample_mode`, `downsample_frac`) without ever touching `validation_df`.
- **Exploratory Data Analysis (`# EDA`)**: Lists available `TARGET_COL` and `feature_set` options from `features.json`, inspects `train_df.head(2)` / `validation_df.head(2)` and `info()`, plots era sizes over time, overlays feature standard-deviation distributions across Train vs. Validation, and displays the top 10 features correlated with `TARGET_COL` along with ADF stationarity checks.

---

### 1. Feature Extraction Parameters (`utils.NumeraiExtractions` — `estimator__extractor__*`)
- Shared across `baseline_unity.ipynb`, `lightgbm_unity.ipynb`, and `minirocket_unity.ipynb` (correlation ranking, variance filtering, sparse selection, PCA, cross-sectional quantization, rolling-era & lag-diff statistics, per-ticker time-series features, and Fourier harmonics).

---

### 2. MiniRocket Model Parameters (`minirocket_utils.MiniRocketNumeraiEstimator` — `estimator__*`)
- **Sequence Construction**:
  - **`input_mode`** (`['ticker_sequence', 'feature_profile']`): `'ticker_sequence'` constructs a strictly causal multivariate `(C, seq_len)` sequence per `numerai_ticker` with a training tail buffer for leak-free validation warm-up; `'feature_profile'` applies univariate MiniRocket across each row's feature vector.
  - **`seq_len`** (`[9, 12, 16, 24]`): Causal lookback window length in eras (minimum `9` to match MiniRocket's length-9 kernels).
  - **`max_channels`** (`[8, 16, 32]`): Maximum top-ranked feature channels fed into multivariate MiniRocket.
  - **`pad_mode`** (`['first', 'neutral']`): Left-padding strategy for tickers with fewer than `seq_len` historical eras (`'first'` repeats earliest value; `'neutral'` pads with `2.0`).
- **MiniRocket Kernel & Dilation Bank (Dempster et al., 2021)**:
  - **`num_kernels`** (`[420, 840, 1680, 3360]`): Number of PPV features (rounded to a multiple of the 84 fixed length-9 `{-1, +2}` kernels).
  - **`max_dilations_per_kernel`** (`[4, 8, 16, 32]`): Maximum exponentially spaced dilations per base kernel.
  - **`fit_sample_size`** (`[1024, 2048, 4096]`): Training subsample size used to draw quantile biases.
  - **`backend`** (`['numpy', 'sktime', 'auto']`): Pure-NumPy vectorized Dempster-2021 MiniRocket (`'numpy'`, zero JIT cold-start & Kaggle-portable) or `sktime` Numba backend (`'sktime'` / `'auto'`).
- **OOM-Proof Batched Linear Head**:
  - **`standardize_features`** (`[True, False]`): Standardizes PPV features before fitting the linear head.
  - **`head_type`** (`['ridge_cv', 'ridge', 'sgd_elasticnet', 'sgd_huber']`): Single-pass batched analytical Ridge / RidgeCV (GCV) or batched SGD head (`elasticnet` / `huber`) — never materializes `(N, num_kernels)` in RAM.
  - **`alpha`** (`[1.0, 10.0, 100.0, 1000.0]` / `Real(1e-2, 1e5, prior='log-uniform')`): Regularization strength for `'ridge'` / SGD heads.
  - **`l1_ratio`** (`[0.05, 0.15, 0.5]` / `Real(0.0, 1.0)`): ElasticNet mixing parameter (active when `head_type='sgd_elasticnet'`).
  - **`huber_epsilon`** (`[1.1, 1.35, 1.75]` / `Real(1.05, 2.5)`): Huber outlier threshold (active when `head_type='sgd_huber'`).
- **Shared `BaseNumeraiEstimator` Controls**: `top_k`, `correlation_method`, `abs_corr`, `era_weighting`, `recency_decay`, `prediction_transform`, `bias_correction`, `walk_forward_step_eras`, `walk_forward_window`, `target_horizon_eras`, `era_stride`.


In [ ]:
import gc
import os
import sys
import json
import time
import warnings
import importlib
from pathlib import Path
from datetime import datetime

python_env_dir = Path(sys.executable).parent
library_bin_dir = python_env_dir / 'Library' / 'bin'
if library_bin_dir.is_dir() and str(library_bin_dir) not in os.environ.get('PATH', ''):
    os.environ['PATH'] = str(library_bin_dir) + os.pathsep + str(python_env_dir) + os.pathsep + os.environ.get('PATH', '')

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
try:
    import seaborn as sns
except ImportError:
    sns = None
from IPython.display import display

warnings.filterwarnings('ignore')

# Local, Kaggle (/kaggle/input, /kaggle/usr/lib, /kaggle/working), & nested dataset path routing
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir).lower() == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)
if os.path.isdir(src_path) and src_path not in sys.path:
    sys.path.insert(0, src_path)

module_search_roots = [
    Path(workspace_root),
    Path('/kaggle/working'),
    Path('/kaggle/usr/lib'),
    Path('/kaggle/input'),
]
for search_root in module_search_roots:
    if not search_root.is_dir():
        continue
    for target_file in ('data_utils.py', 'utils.py', 'baseline_utils.py', 'minirocket_utils.py'):
        for found_path in search_root.rglob(target_file):
            parent_dir = str(found_path.parent)
            if parent_dir not in sys.path:
                sys.path.insert(0, parent_dir)

from pandas.plotting import autocorrelation_plot
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from statsmodels.graphics.gofplots import qqplot

from numerai_tools.scoring import numerai_corr

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

try:
    import wandb
except ImportError:
    wandb = None

try:
    import data_utils
    import utils
    import baseline_utils
    import minirocket_utils
except ModuleNotFoundError as import_err:
    mounted_kaggle_inputs = [str(p) for p in sorted(Path('/kaggle/input').iterdir())] if Path('/kaggle/input').is_dir() else []
    raise ModuleNotFoundError(
        f"{import_err}. Could not locate data_utils.py / utils.py / baseline_utils.py / minirocket_utils.py in "
        f"workspace_root={workspace_root}, /kaggle/working, /kaggle/usr/lib, or /kaggle/input "
        f"(mounted inputs: {mounted_kaggle_inputs}). "
        "On Kaggle, attach a Dataset or Utility Script containing src/data_utils.py, src/utils.py, src/baseline_utils.py, and src/minirocket_utils.py."
    ) from import_err
importlib.reload(data_utils)
importlib.reload(utils)
importlib.reload(baseline_utils)
importlib.reload(minirocket_utils)
from data_utils import compute_per_ticker_timeseries_features, load_unity_numerai_data, load_unity_numerai_data_cached
from utils import (
    NumeraiExtractions,
    build_experiment_setup,
    build_conditional_bayesian_spaces,
    build_quick_results_df,
    candidate_score,
    compute_eda_era_stationarity,
    compute_era_error_envelope,
    plot_error_bin_waterfall,
    configure_error_binning,
    compute_model_error_bin_comparison,
    compute_error_bin_waterfall,
    compute_feature_exposure,
    compute_feature_target_correlations,
    compute_per_era_numerai_corr,
    compute_per_era_spearman_corr,
    deduplicate_param_grid,
    drop_constant_dims,
    evaluate_numerai_predictions,
    export_diagnostics_predictions,
    plot_era_error_envelope,
    prepare_bayesian_space,
    quantize_to_5bins,
    reset_cv_metric_log,
    sharpe_ratio,
    walk_forward_predict,
)
from baseline_utils import BaseNumeraiEstimator, configure_extraction_cache
from minirocket_utils import MiniRocketNumeraiEstimator, SKTIME_MINIROCKET_AVAILABLE

print('Imports loaded successfully')
print(f'workspace_root={workspace_root} | SKOPT_AVAILABLE={SKOPT_AVAILABLE} | SKTIME_MINIROCKET_AVAILABLE={SKTIME_MINIROCKET_AVAILABLE}')

# Setup

In [ ]:
# --- Experiment Knobs (only things you actually change) ---
TARGET_COL = 'target'          # Training target (auto-sets horizon & embargo)
feature_set = 'small'          # 'small', 'medium', or 'all'
era_stride = 4                 # 1, 2, 4 (20d), or 12 (60d)

search_mode = 'bayes'          # 'grid' or 'bayes'
eval_mode = 'both'             # 'static', 'walk_forward', or 'both'
n_splits = 1
cv_split = 0.8                 # Chronological CV train/test split ratio (used when n_splits == 1)
downsample_mode = 'within_era'         # None, 'within_era', 'recent', 'earliest', 'uniform', or 'random_eras'
downsample_frac = 0.2          # Fraction of strided training data to keep in (0.0, 1.0]
n_iter = 1
n_jobs = -1                    # Worker thread count (inner CV stays 1 to avoid RAM duplication)
save_results = True
error_bin_mode = 'pct'         # 'pct' (% error + zero-target tol) or 'abs' (raw residual bins)

setup_cfg = build_experiment_setup(
    target_col=TARGET_COL,
    feature_set=feature_set,
    era_stride=era_stride,
    model_tag='minirocket',
    search_mode=search_mode,
    eval_mode=eval_mode,
    n_splits=n_splits,
    cv_split=cv_split,
    downsample_mode=downsample_mode,
    downsample_frac=downsample_frac,
    n_iter=n_iter,
    n_jobs=1,
    lgbm_n_jobs=n_jobs,
    save_results=save_results,
    error_bin_mode=error_bin_mode,
    workspace_root=workspace_root,
    wandb_project='numerai-minirocket',
)
globals().update(setup_cfg)


# Prep

In [ ]:
_prep_t0 = time.perf_counter()
train_df, validation_df, feature_columns, feature_metadata = load_unity_numerai_data_cached(
    data_version=data_version,
    data_folder=data_folder,
    kaggle_dataset=kaggle_dataset,
    feature_set=feature_set,
    target_col=TARGET_COL,
    era_stride=era_stride,
    downsample_mode=downsample_mode,
    downsample_frac=downsample_frac,
    random_state=random_state,
    ticker_timeseries_features=ticker_timeseries_features,
    ticker_rolling_windows=ticker_rolling_windows,
    ticker_lag_steps=ticker_lag_steps,
    workspace_root=workspace_root,
    auto_download=True,
    use_numerapi_fallback=True,
    force_reload=force_reload_data_cache,
)
prep_elapsed_s = time.perf_counter() - _prep_t0
print(f'Loaded {len(feature_columns)} int8 features | train={train_df.shape} ({train_df[ERA_COL].nunique()} eras) | validation={validation_df.shape} ({validation_df[ERA_COL].nunique()} eras)')

## Exploratory Data Analysis (EDA)

In [ ]:
available_targets = list(feature_metadata.get('targets', [TARGET_COL]))
print(f'Available TARGET_COL options ({len(available_targets)}):', available_targets)
print('Available feature_set options:', {k: len(v) for k, v in feature_metadata.get('feature_sets', {}).items()})

print('\nTraining data:')
display(train_df.head(2))
print('Validation data:')
display(validation_df.head(2))
print('Training set info:')
train_df.info()
print('\nValidation set info:')
validation_df.info()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

train_era_sizes = train_df.groupby(ERA_COL)[TARGET_COL].size()
val_era_sizes = validation_df.groupby(ERA_COL)[TARGET_COL].size()
axes[0].plot(train_era_sizes.index, train_era_sizes.to_numpy(), color='#2563eb', linewidth=1.8, label='Train')
axes[0].plot(val_era_sizes.index, val_era_sizes.to_numpy(), color='#10b981', linewidth=1.8, label='Validation')
axes[0].set_title('Era Sizes Over Time', weight='bold', fontsize=13)
axes[0].set_xlabel('Era')
axes[0].set_ylabel('Rows per Era')
axes[0].legend()

train_feat_stds = train_df[feature_columns].astype(np.float32).std()
val_feat_stds = validation_df[feature_columns].astype(np.float32).std()
axes[1].hist(train_feat_stds, bins=30, alpha=0.55, density=True, color='#2563eb', label='Train')
axes[1].hist(val_feat_stds, bins=30, alpha=0.55, density=True, color='#10b981', label='Validation')
axes[1].set_title('Standard Deviations Over All Features', weight='bold', fontsize=13)
axes[1].set_xlabel('Feature Standard Deviation')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

corr_with_target = train_df[feature_columns].corrwith(train_df[TARGET_COL]).abs().sort_values(ascending=False)
print('Top 10 Features according to absolute correlation with target:')
display(corr_with_target.head(10))

eda_stationarity_df = compute_eda_era_stationarity(train_df, list(corr_with_target.head(10).index), era_col=ERA_COL)
print('ADF Stationarity Check on Top 10 Features (Per-Era Means):')
display(eda_stationarity_df)

In [ ]:
input_columns = [ERA_COL] + ([TICKER_COL] if TICKER_COL in train_df.columns and TICKER_COL in validation_df.columns else []) + feature_columns
X_train = train_df[input_columns].reset_index(drop=True)
y_train = train_df[TARGET_COL].astype(np.float32).reset_index(drop=True)
train_eras = train_df[ERA_COL].astype(np.int32).reset_index(drop=True)

X_validation = validation_df[input_columns].reset_index(drop=True)
y_validation = validation_df[TARGET_COL].astype(np.float32).reset_index(drop=True)
validation_eras = validation_df[ERA_COL].astype(np.int32).reset_index(drop=True)

# Retain only lightweight validation metadata for diagnostics export and free raw DataFrames (~3.8 GB for feature_set='all')
val_diag_meta = validation_df[[c for c in ('id', 'numerai_ticker', 'date', ERA_COL) if c in validation_df.columns]].reset_index(drop=True).copy()
del train_df, validation_df
gc.collect()

# Strictly chronological walk-forward CV splitter (`TimeSeriesSplit` with dynamic `embargo_eras`)
unique_train_eras = np.array(sorted(train_eras.unique()))
if n_splits <= 1:
    split_index = int(len(unique_train_eras) * cv_split)
    era_splits = [(np.arange(0, max(1, split_index - embargo_eras)), np.arange(split_index, len(unique_train_eras)))]
else:
    effective_gap = min(int(embargo_eras), max(0, (len(unique_train_eras) // (n_splits + 1)) - 1))
    era_splits = TimeSeriesSplit(n_splits=n_splits, gap=effective_gap).split(unique_train_eras)
cv_object = [
    (
        np.flatnonzero(train_eras.isin(unique_train_eras[fold_train_eras]).to_numpy()),
        np.flatnonzero(train_eras.isin(unique_train_eras[fold_test_eras]).to_numpy()),
    )
    for fold_train_eras, fold_test_eras in era_splits
]

In [ ]:
raw_feature_sets = feature_metadata.get('feature_sets', {})
unity_feature_groups = {
    str(group_name): [str(c) for c in cols if c in feature_columns]
    for group_name, cols in raw_feature_sets.items()
    if group_name not in ('small', 'medium', 'all') and isinstance(cols, list) and any(c in feature_columns for c in cols)
}
if not unity_feature_groups:
    unity_feature_groups = None

common_extractor = NumeraiExtractions(
    era_col=ERA_COL,
    ticker_col=TICKER_COL,
    target_col=TARGET_COL,
    correlation_method='pandas_mean',
    abs_corr=True,
    feature_groups=unity_feature_groups,
    missing_fill_value=2.0,
)

minirocket_pipeline = Pipeline([
    ('estimator', MiniRocketNumeraiEstimator(
        extractor=common_extractor,
        correlation_method='pandas_mean',
        abs_corr=True,
        era_weighting=era_weighting,
        recency_decay=recency_decay,
        walk_forward_step_eras=None,
        walk_forward_window=walk_forward_window,
        target_horizon_eras=target_horizon_eras,
        era_stride=1,
        era_col=ERA_COL,
        ticker_col=TICKER_COL,
        target_col=TARGET_COL,
        input_mode='ticker_sequence',
        seq_len=12,
        max_channels=16,
        pad_mode='first',
        num_kernels=840,
        max_dilations_per_kernel=8,
        fit_sample_size=2048,
        backend='numpy',
        standardize_features=True,
        head_type='ridge_cv',
        alpha=100.0,
        l1_ratio=0.15,
        huber_epsilon=1.35,
        batch_size=16384,
        n_jobs=lgbm_n_jobs,
        verbose=verbose,
        random_state=random_state,
    )),
])

print('MiniRocket pipeline ready | feature_groups:', 0 if unity_feature_groups is None else len(unity_feature_groups))


# Params

## Grid

### Common Sequence & Feature Extraction Params (NumeraiExtractions)

In [ ]:
# SHARED `NumeraiExtractions` GRID SPACE
GRID_EXTRACTOR_PARAM_SPACE = {
    # ----------------------------------------------------------
    # EXTRACTOR — Era/Target Routing, Stride, Correlation & Feature Selection
    # ----------------------------------------------------------
    'estimator__extractor__era_col': [ERA_COL],
    'estimator__extractor__ticker_col': [TICKER_COL],
    'estimator__extractor__target_col': [
        TARGET_COL,
        # 'target_cyrusd_20',
        # 'target_teager2b_20',
    ],
    'estimator__extractor__year_horizon_filter': [
        None,
        # (1, 2, 3),
        # (4, 5, 6),
        # (7, 8, 9),
    ],
    'estimator__extractor__variance_threshold': [
        None,
        # 0.02,
        # 0.05,
        # 0.10,
    ],
    'estimator__extractor__correlation_method': [
        'pandas_mean',
        # None,
        # 'numerai_mean',
        # 'pearson',
        # 'spearman',
    ],
    'estimator__extractor__abs_corr': [
        True,
        # False,
    ],
    'estimator__extractor__top_k': [
        None,
        # 10,
        # 20,
        # 40,
    ],
    'estimator__extractor__feature_selection_percentile': [
        None,
        # 25.0,
        # 50.0,
        # 75.0,
        # 100.0,
    ],
    'estimator__extractor__sparse_selector': [
        None,
        # 'lasso',
        # 'elasticnet',
    ],
    'estimator__extractor__sparse_alpha': [
        0.01,
        # 0.001,
        # 0.1,
    ],
    'estimator__extractor__pca_n_components': [
        None,
        # 10,
        # 20,
        # 0.90,
    ],
    'estimator__extractor__quantize_scope': [
        'era',
        # 'train_fit',
        # 'global',
    ],
    'estimator__extractor__quantize_strategy': [
        'zscore',
        # 'quantile',
    ],
    'estimator__extractor__sequence_top_n': [
        4,
        # 2,
        # 8,
    ],
    'estimator__extractor__missing_fill_value': [
        2.0,
        # 0.0,
        # 4.0,
        # 'mean',
        # 'median',
        # 'era_mean',
        # 'era_median',
    ],

    # ----------------------------------------------------------
    # EXTRACTOR — Basic Group Stats, Fourier Harmonics & Era/Ticker Sequence Features
    # ----------------------------------------------------------
    'estimator__extractor__basic_group_stats': [
        None,
        # 'mean',
        # 'std',
        # 'skew',
        # 'mean|std',
        # 'mean|skew',
        # 'std|skew',
        # 'mean|std|skew',
    ],
    'estimator__extractor__fourier_period_eras': [
        52,
        # 26,
        # 13,
    ],
    'estimator__extractor__fourier_order': [
        None,
        # 1,
        # 2,
        # 3,
    ],
    'estimator__extractor__rolling_era_windows': [
        (4, 12),
        # (2, 6, 12),
        # (4, 8, 16),
    ],
    'estimator__extractor__include_rolling_era': [
        False,
        # True,
    ],
    'estimator__extractor__era_lag_scales': [
        (1, 2, 4),
        # (1, 2, 4, 8),
        # (2, 4, 8),
    ],
    'estimator__extractor__include_lag_diff': [
        False,
        # True,
    ],
    'estimator__extractor__ticker_features': [
        None,
        # 'rolling_mean',
        # 'momentum_diff',
        # 'weeks_since_high',
        # 'rolling_rank',
        # 'weeks_since_change',
        # 'expanding_stats',
        # 'raw_lags',
        # 'rolling_mean|momentum_diff',
        # 'rolling_mean|momentum_diff|rolling_rank',
        # 'expanding_stats|raw_lags',
        # 'rolling_mean|momentum_diff|weeks_since_high|rolling_rank|weeks_since_change|expanding_stats|raw_lags',
    ],
    'estimator__extractor__ticker_windows': [
        (4, 12),
        # (2, 6, 12),
    ],
    'estimator__extractor__ticker_lag_steps': [
        None,
        # (1, 2),
        # (1, 2, 4),
    ],
    'estimator__extractor__era_dispersion_features': [
        None,
        # 'mean',
        # 'std',
        # 'extreme_frac',
        # 'mean|std',
        # 'mean|extreme_frac',
        # 'std|extreme_frac',
        # 'mean|std|extreme_frac',
    ],
}

### MiniRocket Model Params (`MiniRocketNumeraiEstimator`)


In [ ]:
GRID_PARAM_SPACE_MINIROCKET = {
    **GRID_EXTRACTOR_PARAM_SPACE,
    # BaseNumeraiEstimator shared params
    'estimator__top_k': [None, 20, 40],
    'estimator__correlation_method': ['pandas_mean', None],
    'estimator__abs_corr': [True, False],
    'estimator__era_weighting': [None, 'equal_era', 'recency_exp'],
    'estimator__recency_decay': [0.005],
    'estimator__prediction_transform': [None, 'era_rank', 'era_zscore'],
    'estimator__bias_correction': [None, 'mean_residual'],
    'estimator__walk_forward_step_eras': [None, 20],
    'estimator__walk_forward_window': ['expanding', 'sliding'],
    'estimator__target_horizon_eras': [target_horizon_eras],
    'estimator__era_stride': [1, 2],
    'estimator__era_col': [ERA_COL],
    'estimator__ticker_col': [TICKER_COL],
    'estimator__target_col': [TARGET_COL],
    'estimator__verbose': [verbose],
    'estimator__random_state': [random_state],
    # MiniRocket sequence & transform params
    'estimator__input_mode': ['ticker_sequence', 'feature_profile'],
    'estimator__seq_len': [9, 12, 16],
    'estimator__max_channels': [8, 16],
    'estimator__pad_mode': ['first', 'neutral'],
    'estimator__num_kernels': [420, 840],
    'estimator__max_dilations_per_kernel': [4, 8],
    'estimator__fit_sample_size': [2048],
    'estimator__backend': ['numpy'],
    # Batched linear head params
    'estimator__standardize_features': [True],
    'estimator__head_type': ['ridge_cv', 'ridge'],
    'estimator__alpha': [10.0, 100.0, 1000.0],
    'estimator__l1_ratio': [0.15],
    'estimator__huber_epsilon': [1.35],
    'estimator__n_jobs': [lgbm_n_jobs],
}
print(f'GRID_PARAM_SPACE_MINIROCKET keys={len(GRID_PARAM_SPACE_MINIROCKET)}')


## Bayesian

### Common Sequence & Feature Extraction Params (NumeraiExtractions)

In [ ]:
# BAYESIAN SEARCH ONLY — Shared `BAYESIAN_EXTRACTOR_PARAM_SPACE`
# Mode/selection parameters start with `None` (off); boolean `include_*` flags use `[False, True]`.
BAYESIAN_EXTRACTOR_PARAM_SPACE = dict()
if SKOPT_AVAILABLE:
    BAYESIAN_EXTRACTOR_PARAM_SPACE = {
        # ----------------------------------------------------------
        # EXTRACTOR — Era/Target Routing, Stride, Correlation & Feature Selection
        # ----------------------------------------------------------
        'estimator__extractor__era_col': Categorical([ERA_COL]),
        'estimator__extractor__ticker_col': Categorical([TICKER_COL]),
        'estimator__extractor__target_col': Categorical([
            TARGET_COL,
        ]),
        'estimator__extractor__year_horizon_filter': Categorical([
            None,
            (1, 2, 3),
            (4, 5, 6),
            (7, 8, 9),
        ]),
        'estimator__extractor__variance_threshold': Categorical([
            None,
            0.02,
            0.05,
            0.10,
        ]),
        'estimator__extractor__correlation_method': Categorical([
            None,
            'pandas_mean',
            'numerai_mean',
            'pearson',
            'spearman',
        ]),
        'estimator__extractor__abs_corr': Categorical([True, False]),
        'estimator__extractor__top_k': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__extractor__feature_selection_percentile': Categorical([
            None,
            25.0,
            50.0,
            75.0,
            100.0,
        ]),
        'estimator__extractor__sparse_selector': Categorical([
            None,
            'lasso',
            'elasticnet',
        ]),
        'estimator__extractor__sparse_alpha': Real(1e-4, 1e-1, prior='log-uniform'),
        'estimator__extractor__pca_n_components': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__extractor__quantize_scope': Categorical([
            'era',
            'train_fit',
            'global',
        ]),
        'estimator__extractor__quantize_strategy': Categorical([
            'zscore',
            'quantile',
        ]),
        'estimator__extractor__sequence_top_n': Categorical([
            2,
            4,
            8,
        ]),
        'estimator__extractor__missing_fill_value': Categorical([
            2.0,
            0.0,
            4.0,
            'mean',
            'median',
            'era_mean',
            'era_median',
        ]),

        # ----------------------------------------------------------
        # EXTRACTOR — Basic Group Stats, Fourier Harmonics & Era/Ticker Sequence Features
        # ----------------------------------------------------------
        'estimator__extractor__basic_group_stats': Categorical([
            None,
            'mean',
            'std',
            'skew',
            'mean|std',
            'mean|skew',
            'std|skew',
            'mean|std|skew',
        ]),
        'estimator__extractor__fourier_period_eras': Categorical([
            52,
            26,
            13,
        ]),
        'estimator__extractor__fourier_order': Categorical([
            None,
            1,
            2,
            3,
        ]),
        'estimator__extractor__rolling_era_windows': Categorical([
            (4, 12),
            (2, 6, 12),
            (4, 8, 16),
        ]),
        'estimator__extractor__include_rolling_era': Categorical([False, True]),
        'estimator__extractor__era_lag_scales': Categorical([
            (1, 2, 4),
            (1, 2, 4, 8),
            (2, 4, 8),
        ]),
        'estimator__extractor__include_lag_diff': Categorical([False, True]),
        'estimator__extractor__ticker_features': Categorical([
            None,
            'rolling_mean',
            'momentum_diff',
            'weeks_since_high',
            'rolling_rank',
            'weeks_since_change',
            'expanding_stats',
            'raw_lags',
            'rolling_mean|momentum_diff',
            'rolling_mean|momentum_diff|rolling_rank',
            'expanding_stats|raw_lags',
            'rolling_mean|momentum_diff|weeks_since_high|rolling_rank|weeks_since_change|expanding_stats|raw_lags',
        ]),
        'estimator__extractor__ticker_windows': Categorical([
            (4, 12),
            (2, 6, 12),
        ]),
        'estimator__extractor__ticker_lag_steps': Categorical([
            None,
            (1, 2),
            (1, 2, 4),
        ]),
        'estimator__extractor__era_dispersion_features': Categorical([
            None,
            'mean',
            'std',
            'extreme_frac',
            'mean|std',
            'mean|extreme_frac',
            'std|extreme_frac',
            'mean|std|extreme_frac',
        ]),
    }

### MiniRocket Model Params (`MiniRocketNumeraiEstimator`)


In [ ]:
BAYESIAN_PARAM_SPACE_MINIROCKET = dict()
if SKOPT_AVAILABLE:
    BAYESIAN_PARAM_SPACE_MINIROCKET = prepare_bayesian_space({
        **BAYESIAN_EXTRACTOR_PARAM_SPACE,
        # BaseNumeraiEstimator shared params
        'estimator__top_k': Categorical([None, 16, 32, 64]),
        'estimator__correlation_method': Categorical([None, 'pandas_mean', 'numerai_mean', 'pearson']),
        'estimator__abs_corr': Categorical([True, False]),
        'estimator__era_weighting': Categorical([None, 'equal_era', 'recency_linear', 'recency_exp']),
        'estimator__recency_decay': Real(1e-4, 0.05, prior='log-uniform'),
        'estimator__prediction_transform': Categorical([None, 'era_rank', 'era_zscore']),
        'estimator__bias_correction': Categorical([None, 'mean_residual']),
        'estimator__walk_forward_step_eras': Categorical([None, 10, 20, 40]),
        'estimator__walk_forward_window': Categorical(['expanding', 'sliding']),
        'estimator__target_horizon_eras': Categorical([target_horizon_eras]),
        'estimator__era_stride': Categorical([1, 2, 4]),
        'estimator__era_col': Categorical([ERA_COL]),
        'estimator__ticker_col': Categorical([TICKER_COL]),
        'estimator__target_col': Categorical([TARGET_COL]),
        'estimator__verbose': Categorical([verbose]),
        'estimator__random_state': Categorical([random_state]),
        # MiniRocket sequence & transform params
        'estimator__input_mode': Categorical(['ticker_sequence', 'feature_profile']),
        'estimator__seq_len': Categorical([9, 12, 16, 24]),
        'estimator__max_channels': Categorical([8, 16, 32]),
        'estimator__pad_mode': Categorical(['first', 'neutral']),
        'estimator__num_kernels': Categorical([420, 840, 1680]),
        'estimator__max_dilations_per_kernel': Categorical([4, 8, 16]),
        'estimator__fit_sample_size': Categorical([1024, 2048, 4096]),
        'estimator__backend': Categorical(['numpy']),
        # Batched linear head params
        'estimator__standardize_features': Categorical([True, False]),
        'estimator__head_type': Categorical(['ridge_cv', 'ridge', 'sgd_elasticnet', 'sgd_huber']),
        'estimator__alpha': Real(1e-2, 1e5, prior='log-uniform'),
        'estimator__l1_ratio': Real(0.0, 1.0),
        'estimator__huber_epsilon': Real(1.05, 2.5),
        'estimator__n_jobs': Categorical([lgbm_n_jobs]),
    })
print(f'BAYESIAN_PARAM_SPACE_MINIROCKET keys={len(BAYESIAN_PARAM_SPACE_MINIROCKET)}')


## Active Search Space Validation & Routing

In [ ]:
minirocket_param_names = set(minirocket_pipeline.get_params(deep=True))
expected_tunable_params = {
    p for p in minirocket_param_names
    if p.startswith('estimator__') and p not in ('estimator__extractor', 'estimator__extractor__feature_groups', 'estimator__batch_size')
}
assert set(GRID_PARAM_SPACE_MINIROCKET).issubset(minirocket_param_names)
assert expected_tunable_params == set(GRID_PARAM_SPACE_MINIROCKET), (
    f'Missing from GRID_PARAM_SPACE_MINIROCKET: {sorted(expected_tunable_params - set(GRID_PARAM_SPACE_MINIROCKET))}'
)
if SKOPT_AVAILABLE:
    assert set(BAYESIAN_PARAM_SPACE_MINIROCKET).issubset(minirocket_param_names)
    assert expected_tunable_params == set(BAYESIAN_PARAM_SPACE_MINIROCKET), (
        f'Missing from BAYESIAN_PARAM_SPACE_MINIROCKET: {sorted(expected_tunable_params - set(BAYESIAN_PARAM_SPACE_MINIROCKET))}'
    )

if search_mode == 'bayes':
    _pruned_mr_bayes = drop_constant_dims(BAYESIAN_PARAM_SPACE_MINIROCKET, minirocket_pipeline)
    minirocket_param_space = build_conditional_bayesian_spaces(_pruned_mr_bayes, total_n_iter=n_iter)
else:
    minirocket_param_space = GRID_PARAM_SPACE_MINIROCKET
_active_dim_count = len(set().union(*(sub.keys() for sub, _ in minirocket_param_space))) if isinstance(minirocket_param_space, list) else len(minirocket_param_space)
print(f'Active search_mode={search_mode} | Active MiniRocket + Extractor dims={_active_dim_count} (total exposed={len(expected_tunable_params)})')


# Train

In [ ]:
_train_t0 = time.perf_counter()
model_results = {}
search_summary_rows = []
cv_results_by_model = {}
fitted_default_models = {}

effective_cache_max_entries = (
    min(int(extraction_cache_max_entries), 2)
    if str(feature_set).strip().lower() in ('medium', 'all')
    else int(extraction_cache_max_entries)
)
configure_extraction_cache(enabled=extraction_cache_enabled, max_entries=effective_cache_max_entries)

model_name = 'MiniRocket Regressor'
print(
    f'Running {model_name} inline (n_jobs={lgbm_n_jobs}, inner cv_n_jobs=1, '
    f'extraction_cache_max_entries={effective_cache_max_entries})'
)
default_model = clone(minirocket_pipeline).fit(X_train, y_train)
fitted_default_models[model_name] = default_model

train_preds = default_model.predict(X_train)
val_preds = default_model.predict(X_validation)

train_eval = evaluate_numerai_predictions(
    y_train,
    train_preds,
    eras=train_eras,
    features_df=X_train,
    feature_columns=feature_columns,
    label=f'{model_name} — Final Training Scores',
    verbose=True,
)
val_eval = evaluate_numerai_predictions(
    y_validation,
    val_preds,
    eras=validation_eras,
    features_df=X_validation,
    feature_columns=feature_columns,
    label=f'{model_name} — Final Validation Scores',
    verbose=True,
)

default_variant_name = f'{model_name} [Default]'
if eval_mode in ('static', 'both'):
    model_results[default_variant_name] = {
        'variant': default_variant_name,
        'train_corr': train_eval['numerai_corr'],
        'validation_corr': val_eval['numerai_corr'],
        'validation_corr_pct': val_eval['numerai_corr_pct'],
        'positive_era_pct': val_eval['positive_era_pct'],
        'directional_accuracy_pct': val_eval['directional_accuracy_pct'],
        'validation_spearman': val_eval['spearman'],
        'validation_payout': val_eval['payout'],
        'validation_sharpe': val_eval['numerai_sharpe'],
        'validation_spearman_sharpe': val_eval['spearman_sharpe'],
        'validation_mae': val_eval['mae'],
        'forecast_bias': val_eval['forecast_bias'],
        'directional_hit_rate': val_eval['directional_hit_rate'],
        'era_corr_lag1_acf': val_eval['era_corr_lag1_acf'],
        'ljung_box_pvalue': val_eval['ljung_box_pvalue'],
        'arch_lm_pvalue': val_eval['arch_lm_pvalue'],
        'adf_pvalue': val_eval['adf_pvalue'],
        'era_corr_skew': val_eval['era_corr_skew'],
        'era_corr_kurtosis': val_eval['era_corr_kurtosis'],
        'jarque_bera_pvalue': val_eval['jarque_bera_pvalue'],
        'feature_exposure': val_eval['feature_exposure'],
        'max_drawdown': val_eval['max_drawdown'],
        'corr_gap': train_eval['numerai_corr'] - val_eval['numerai_corr'],
        'sanity_status': val_eval['sanity_status'],
        'validation_preds': val_preds,
        'validation_era_corrs': val_eval['numerai_era_corrs'],
        'params': default_model.named_steps['estimator'].get_params(),
    }

if eval_mode in ('walk_forward', 'both'):
    wf_preds = walk_forward_predict(
        minirocket_pipeline,
        X_train,
        y_train,
        train_eras,
        X_validation,
        y_validation,
        validation_eras,
        step_eras=walk_forward_step_eras,
        window_mode=walk_forward_window,
        target_horizon_eras=target_horizon_eras,
        era_stride=era_stride,
    )
    wf_eval = evaluate_numerai_predictions(
        y_validation,
        wf_preds,
        eras=validation_eras,
        features_df=X_validation,
        feature_columns=feature_columns,
        label=f'{model_name} — Walk-Forward Validation Scores',
        verbose=True,
    )
    wf_variant_name = f'{model_name} [Walk-Forward]'
    model_results[wf_variant_name] = {
        'variant': wf_variant_name,
        'train_corr': train_eval['numerai_corr'],
        'validation_corr': wf_eval['numerai_corr'],
        'validation_corr_pct': wf_eval['numerai_corr_pct'],
        'positive_era_pct': wf_eval['positive_era_pct'],
        'directional_accuracy_pct': wf_eval['directional_accuracy_pct'],
        'validation_spearman': wf_eval['spearman'],
        'validation_payout': wf_eval['payout'],
        'validation_sharpe': wf_eval['numerai_sharpe'],
        'validation_spearman_sharpe': wf_eval['spearman_sharpe'],
        'validation_mae': wf_eval['mae'],
        'forecast_bias': wf_eval['forecast_bias'],
        'directional_hit_rate': wf_eval['directional_hit_rate'],
        'era_corr_lag1_acf': wf_eval['era_corr_lag1_acf'],
        'ljung_box_pvalue': wf_eval['ljung_box_pvalue'],
        'arch_lm_pvalue': wf_eval['arch_lm_pvalue'],
        'adf_pvalue': wf_eval['adf_pvalue'],
        'era_corr_skew': wf_eval['era_corr_skew'],
        'era_corr_kurtosis': wf_eval['era_corr_kurtosis'],
        'jarque_bera_pvalue': wf_eval['jarque_bera_pvalue'],
        'feature_exposure': wf_eval['feature_exposure'],
        'max_drawdown': wf_eval['max_drawdown'],
        'corr_gap': train_eval['numerai_corr'] - wf_eval['numerai_corr'],
        'sanity_status': wf_eval['sanity_status'],
        'validation_preds': wf_preds,
        'validation_era_corrs': wf_eval['numerai_era_corrs'],
        'params': default_model.named_steps['estimator'].get_params(),
    }

if search_mode == 'bayes':
    if not SKOPT_AVAILABLE or BayesSearchCV is None:
        raise ImportError('scikit-optimize is required for Bayesian search')
    minirocket_search_spaces = (
        minirocket_param_space
        if isinstance(minirocket_param_space, list)
        else build_conditional_bayesian_spaces(minirocket_param_space, total_n_iter=n_iter)
    )
    search = BayesSearchCV(
        estimator=minirocket_pipeline,
        search_spaces=minirocket_search_spaces,
        n_iter=n_iter,
        optimizer_kwargs={'base_estimator': 'ET', 'acq_func': 'EI', 'n_initial_points': min(4, max(1, n_iter // 3))},
        scoring=candidate_score,
        cv=cv_object,
        refit=False,
        random_state=random_state,
        verbose=verbose,
        error_score=error_score,
        n_jobs=1,
    )
else:
    deduped_grid = deduplicate_param_grid(minirocket_param_space, base_params=minirocket_pipeline.get_params(deep=True))
    search = GridSearchCV(
        estimator=minirocket_pipeline,
        param_grid=deduped_grid,
        scoring=candidate_score,
        cv=cv_object,
        refit=False,
        verbose=verbose,
        error_score=error_score,
        n_jobs=1,
    )

reset_cv_metric_log()
search.fit(X_train, y_train, groups=train_eras)
cv_results_by_model[model_name] = build_quick_results_df(search)
search_summary_rows.append({
    'model': model_name,
    'search_mode': search_mode,
    'best_cv_corr': float(search.best_score_),
    'best_cv_corr_pct': round(float(search.best_score_) * 100.0, 2),
    'best_params': dict(search.best_params_),
})
train_elapsed_s = time.perf_counter() - _train_t0
print(f'Completed MiniRocket train + CV in {train_elapsed_s:.1f}s')

In [ ]:
comparison_columns = [
    'variant',
    'train_corr',
    'validation_corr',
    'validation_corr_pct',
    'positive_era_pct',
    'directional_accuracy_pct',
    'validation_spearman',
    'validation_payout',
    'validation_sharpe',
    'validation_spearman_sharpe',
    'validation_mae',
    'forecast_bias',
    'directional_hit_rate',
    'era_corr_lag1_acf',
    'ljung_box_pvalue',
    'arch_lm_pvalue',
    'adf_pvalue',
    'era_corr_skew',
    'era_corr_kurtosis',
    'jarque_bera_pvalue',
    'feature_exposure',
    'max_drawdown',
    'corr_gap',
    'sanity_status',
]
comparison_df = (
    pd.DataFrame([{column: result[column] for column in comparison_columns} for result in model_results.values()])
    .sort_values(['validation_sharpe', 'validation_corr'], ascending=False, kind='stable')
    .reset_index(drop=True)
)
search_summary_df = (
    pd.DataFrame(search_summary_rows)
    .sort_values('best_cv_corr', ascending=False, kind='stable')
    .reset_index(drop=True)
)
best_variant_name = str(comparison_df.loc[0, 'variant'])
best_validation_era_corrs = model_results[best_variant_name]['validation_era_corrs']
best_validation_corr = model_results[best_variant_name]['validation_corr']

print('MiniRocket Holdout Validation Comparison (Official Numerai CORR + Tutorial Spearman/Payout/Sharpe/MAE/Feature Exposure):')
display(comparison_df)
print(f'CV Search Summary ({search_mode}, refit=False):')
display(search_summary_df)
print(f'Top Holdout Variant: {best_variant_name}')

# Analysis

In [ ]:
era_positions = np.arange(len(best_validation_era_corrs))
tick_step = max(1, len(best_validation_era_corrs) // 10)
tick_labels = best_validation_era_corrs.index.astype(str)[::tick_step]
line_colors = ['#2563eb', '#10b981', '#f59e0b', '#8b5cf6']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.bar(
    era_positions,
    best_validation_era_corrs.to_numpy(),
    color=['#10b981' if era_corr >= 0 else '#ef4444' for era_corr in best_validation_era_corrs.to_numpy()],
    alpha=0.85,
)
ax.axhline(0.0, color='#64748b', linewidth=1.0)
ax.axhline(best_validation_corr, color='#1d4ed8', linestyle='--', linewidth=1.5, label=f'Winner Mean = {best_validation_corr:.4f}')
ax.set_xticks(era_positions[::tick_step])
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Validation Era')
ax.set_ylabel('Numerai CORR')
ax.set_title(f'Per-Era Validation CORR ({best_variant_name})')
ax.legend()

ax = axes[1]
for color_index, (variant_name, result) in enumerate(model_results.items()):
    line_color = line_colors[color_index % len(line_colors)]
    ax.plot(
        era_positions,
        result['validation_era_corrs'].cumsum().to_numpy(),
        color=line_color,
        linewidth=2.0,
        label=f"{variant_name} (Sharpe {result['validation_sharpe']:.2f})",
    )
ax.axhline(0.0, color='#64748b', linewidth=0.8)
ax.set_xticks(era_positions[::tick_step])
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Validation Era')
ax.set_ylabel('Cumulative CORR')
ax.set_title('Cumulative Validation CORR Across LightGBM Variants')
ax.legend(fontsize=8)

plt.tight_layout()
if save_results:
    results_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(results_dir / f'{timestamp}_{experiment_name}_era_perf.png', dpi=140)
    comparison_df.to_csv(results_dir / f'{timestamp}_{experiment_name}_comparison.csv', index=False)
    search_summary_df.to_csv(results_dir / f'{timestamp}_{experiment_name}_{search_mode}_summary.csv', index=False)
if export_diagnostics or save_results:
    results_dir.mkdir(parents=True, exist_ok=True)
    diagnostics_parquet_path = export_diagnostics_predictions(
        val_diag_meta,
        model_results[best_variant_name]['validation_preds'],
        results_dir / 'validation_predictions.parquet',
        era_col=ERA_COL,
    )
    print('Exported Numerai Diagnostics predictions Parquet (MMC & FNC):', diagnostics_parquet_path)
plt.show()

# Brownlee Residual, White-Noise & Tail-Normality Diagnostics (Per-Era Mean Residual + Era CORR Autocorrelation + Residual Q-Q Plot)
winner_val_preds = np.asarray(model_results[best_variant_name]['validation_preds'], dtype=np.float64).ravel()
winner_residuals = pd.Series(y_validation.to_numpy(dtype=np.float64) - winner_val_preds, index=validation_eras.index)
per_era_mean_residuals = winner_residuals.groupby(validation_eras, sort=True).mean()

fig_res, axes_res = plt.subplots(1, 3, figsize=(20, 4.5))
axes_res[0].plot(era_positions, per_era_mean_residuals.to_numpy(), color='#dc2626', marker='o', markersize=3.5, linewidth=1.6, label='Mean Residual (y - y_hat)')
axes_res[0].axhline(0.0, color='#64748b', linestyle='--', linewidth=1.2, label='Zero Bias (0.0)')
axes_res[0].set_xticks(era_positions[::tick_step])
axes_res[0].set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
axes_res[0].set_xlabel('Validation Era')
axes_res[0].set_ylabel('Mean Forecast Residual')
axes_res[0].set_title(f'Per-Era Mean Forecast Residual ({best_variant_name})')
axes_res[0].legend()

if len(best_validation_era_corrs) >= 2:
    autocorrelation_plot(best_validation_era_corrs, ax=axes_res[1], color='#2563eb')
axes_res[1].set_title(f'White-Noise Check: Per-Era CORR Autocorrelation ({best_variant_name})')

if len(per_era_mean_residuals) >= 3:
    qqplot(per_era_mean_residuals.to_numpy(dtype=np.float64), line='45', fit=True, ax=axes_res[2])
axes_res[2].set_title(f'Residual Q-Q Plot ({best_variant_name})')
plt.tight_layout()
if save_results:
    plt.savefig(results_dir / f'{timestamp}_{experiment_name}_residual_white_noise.png', dpi=140)
plt.show()

In [ ]:
fitted_mr = fitted_default_models[model_name].named_steps['estimator']
top_kernels_df = fitted_mr.kernel_importance_df_.head(20)
top_channels_df = fitted_mr.channel_importance_df_.head(20)
print(
    f'MiniRocket Fitted Summary | backend={fitted_mr.active_backend_} | '
    f'input_mode={fitted_mr.active_input_mode_} | channels={fitted_mr.n_channels_in_} | '
    f'seq_len={fitted_mr.seq_len_in_} | rocket_features={fitted_mr.n_rocket_features_} | '
    f'head={fitted_mr.active_head_type_} (alpha={fitted_mr.chosen_alpha_:.4g})'
)
print('Top 20 MiniRocket PPV Kernels by |Weight|:')
display(top_kernels_df)
print('Top 20 Input Channels by Aggregated MiniRocket Kernel Weight:')
display(top_channels_df)

if plt is not None and len(top_kernels_df) and len(top_channels_df):
    fig_imp, axes_imp = plt.subplots(1, 2, figsize=(16, 6))
    plot_k = top_kernels_df.sort_values('abs_coef', ascending=True)
    axes_imp[0].barh(plot_k['rocket_feature'], plot_k['abs_coef'], color='#2563eb', alpha=0.85)
    axes_imp[0].set_xlabel('|Standardized Linear Weight|')
    axes_imp[0].set_title('Top 20 MiniRocket PPV Kernels')

    plot_c = top_channels_df.sort_values('kernel_weight_sum', ascending=True)
    axes_imp[1].barh(plot_c['feature'], plot_c['kernel_weight_sum'], color='#059669', alpha=0.85)
    axes_imp[1].set_xlabel('Aggregated |Kernel Weight| Attribution')
    axes_imp[1].set_title('Top 20 Input Channels (Features)')
    plt.tight_layout()
    if save_results:
        results_dir.mkdir(parents=True, exist_ok=True)
        plt.savefig(results_dir / f'{timestamp}_{experiment_name}_minirocket_importance.png', dpi=140)
    plt.show()


In [ ]:
for cur_model_name, cv_results_df in cv_results_by_model.items():
    print(f'Top CV candidates for {cur_model_name}:')
    display(cv_results_df.head(10))
    if save_results:
        results_dir.mkdir(parents=True, exist_ok=True)
        cv_results_path = results_dir / f'{timestamp}_{experiment_name}_{search_mode}_cv.csv'
        cv_results_df.to_csv(cv_results_path, index=False)
        print('Saved CV results CSV:', cv_results_path)

## Error-Direction Envelope Analysis (All Variants — Best Highlighted)

In [ ]:
envelope_dfs = {}
for variant_name in comparison_df['variant'].tolist():
    is_best_variant = (variant_name == best_variant_name)
    variant_slug = (
        variant_name.lower()
        .replace(' ', '_')
        .replace('[', '')
        .replace(']', '')
        .replace('-', '_')
    )
    envelope_save_path = (
        results_dir / f'{timestamp}_{experiment_name}_{variant_slug}_error_envelope.png'
        if save_results
        else None
    )
    envelope_dfs[variant_name] = plot_era_error_envelope(
        y_validation,
        model_results[variant_name]['validation_preds'],
        validation_eras,
        title=variant_name,
        band_size=0.15,
        is_best=is_best_variant,
        save_path=envelope_save_path,
    )

envelope_df = envelope_dfs[best_variant_name]
print(f'Best Model Envelope Summary ({best_variant_name}):')
display(envelope_df.head(10))

## Error-Bin Waterfall & Financial Quarter Breakdown

In [ ]:
val_dates = val_diag_meta['date'] if 'date' in val_diag_meta.columns else None
model_error_bin_comparison_df = compute_model_error_bin_comparison(
    model_results,
    y_validation,
    best_variant_name=best_variant_name,
    mode=error_bin_mode,
    pct_bins=error_pct_bins,
    zero_tols=error_zero_tols,
    abs_bins=error_abs_bins,
)
print('Overall Error-Bin Comparison Across Evaluated Models:')
display(model_error_bin_comparison_df)

waterfall_df = compute_error_bin_waterfall(
    y_validation,
    model_results[best_variant_name]['validation_preds'],
    validation_eras,
    dates=val_dates,
    mode=error_bin_mode,
    pct_bins=error_pct_bins,
    zero_tols=error_zero_tols,
    abs_bins=error_abs_bins,
)
print(f'Error-Bin Waterfall Summary Table ({best_variant_name}):')
display(waterfall_df)

waterfall_png_path = (
    results_dir / f'{timestamp}_{experiment_name}_error_bin_waterfall.png'
    if save_results
    else None
)
plot_error_bin_waterfall(
    waterfall_df,
    title=best_variant_name,
    save_path=waterfall_png_path,
)
if save_results:
    results_dir.mkdir(parents=True, exist_ok=True)
    waterfall_csv_path = results_dir / f'{timestamp}_{experiment_name}_error_bin_waterfall.csv'
    waterfall_df.to_csv(waterfall_csv_path, index=False)
    model_error_bin_comparison_df.to_csv(
        results_dir / f'{timestamp}_{experiment_name}_model_error_bins.csv', index=False
    )
    print('Saved Error-Bin Waterfall CSV:', waterfall_csv_path)

## Optional Experiment Tracking (Weights & Biases)

In [ ]:
if use_wandb:
    if wandb is None:
        print('Skipping W&B logging: wandb is not installed.')
    else:
        wandb_run = wandb.init(
            project=wandb_project,
            name=experiment_name,
            config={
                'data_version': data_version,
                'era_tag': era_tag,
                'model_tag': model_tag,
                'TARGET_COL': TARGET_COL,
                'feature_set': feature_set,
                'era_stride': era_stride,
                'search_mode': search_mode,
                'cv_strategy': cv_strategy,
                'n_splits': n_splits,
                'cv_split': cv_split,
                'downsample_mode': downsample_mode,
                'downsample_frac': downsample_frac,
                'target_horizon_eras': target_horizon_eras,
                'embargo_eras': embargo_eras,
                'eval_mode': eval_mode,
                'walk_forward_step_eras': walk_forward_step_eras,
                'walk_forward_window': walk_forward_window,
                'era_weighting': era_weighting,
                'recency_decay': recency_decay,
                'random_state': random_state,
                'n_iter': n_iter,
                'verbose': verbose,
                'n_jobs': n_jobs,
                'lgbm_n_jobs': lgbm_n_jobs,
                'error_bin_mode': error_bin_mode,
            },
        )
        log_payload = {
            'comparison_table': wandb.Table(dataframe=comparison_df),
            'search_summary_table': wandb.Table(dataframe=search_summary_df.drop(columns=['best_params'], errors='ignore')),
            'timing/prep_elapsed_s': float(prep_elapsed_s),
            'timing/train_elapsed_s': float(train_elapsed_s),
        }
        for m_name, m_cv_df in cv_results_by_model.items():
            m_slug = m_name.lower().replace(' ', '_')
            log_payload[f'cv_table/{m_slug}'] = wandb.Table(dataframe=m_cv_df.astype(str))
        for row in search_summary_rows:
            m_slug = row['model'].lower().replace(' ', '_')
            log_payload[f'best_cv_corr/{m_slug}'] = float(row['best_cv_corr'])
        wandb.log(log_payload)
        wandb.finish()
        print('Logged experiment tables and metrics to Weights & Biases.')
else:
    print('W&B tracking disabled (use_wandb=False).')